
# 1. Le GPU et le choix du modèle

Le modèle est téléchargé au premier lancement dans HF_HOME. On le place dans ~/work, le dossier persistant du service, pour ne pas le retélécharger à chaque redémarrage. Cette ligne doit être exécutée avant tout import de transformers.


In [4]:

%pip install -q -U "transformers>=4.57" accelerate s3fs pillow pandas mlflow

Note: you may need to restart the kernel to use updated packages.


In [1]:
%pip install -q -U OPENAI

Note: you may need to restart the kernel to use updated packages.


In [31]:
import os
for k in sorted(os.environ):
    if "OPENAI" in k or "API" in k or"LLM" in k:
        print(k)

OPENAI_API_KEY


In [32]:
import os
import requests

BASE_URL = "https://llm.lab.sspcloud.fr/api"
API_KEY = os.environ["OPENAI_API_KEY"]

response = requests.get(
    f"{BASE_URL}/models",
    headers={"Authorization": f"Bearer {API_KEY}"},
    timeout=30,
)
response.raise_for_status()

for m in response.json()["data"]:
    print(m["id"])

gemma4-26b-moe
qwen3-6-35b-moe
qwen3-vl
qwen3-embedding-8b
chandra-ocr-2
qwen3-8-27b


# 2. Charger le modèle

Le premier chargement télécharge les poids : plusieurs minutes. Les suivants sont rapides.


In [41]:
import os
import io
import base64
import requests
import time
import pandas as pd

BASE_URL = "https://llm.lab.sspcloud.fr/api"
HEADERS = {"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"}
MODELE = "qwen3-vl"  

# 3. Interroger le modèle

do_sample=False : le modèle répond toujours la même chose à la même question. Indispensable pour mesurer sinon deux exécutions donnent deux taux différents.


In [33]:
def demander(image, question, max_new_tokens=400):
    """Envoie une image PIL et une question au modèle via l'API, renvoie le texte produit."""
    image = image.copy()
    image.thumbnail((1400, 1400))
    tampon = io.BytesIO()
    image.convert("RGB").save(tampon, format="PNG")
    image_b64 = base64.b64encode(tampon.getvalue()).decode()

    payload = {
        "model": MODELE,
        "messages": [{"role": "user", "content": [
            {"type": "image_url",
             "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
            {"type": "text", "text": question},
        ]}],
        "max_tokens": max_new_tokens,
        "temperature": 0,
    }
    r = requests.post(f"{BASE_URL}/chat/completions", headers=HEADERS,
                      json=payload, timeout=300)
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"].strip()

# 4. Les données

Mêmes images que le notebook 01. On garde la métropole : c'est le seul périmètre comparable au Fiqual de type 1.


In [34]:
import io
from pathlib import Path

import pandas as pd
import s3fs
from PIL import Image

BUCKET = "projet-mesure-qualite-rp"
CAMPAGNE = "RG25"
fs = s3fs.S3FileSystem()

fichiers = [f for f in fs.find(f"{BUCKET}/data/raw/{CAMPAGNE}/BIE") if f.endswith("_LOG.jpg")]
logs = pd.DataFrame({"chemin": fichiers})
logs["nom"] = logs.chemin.map(lambda c: Path(c).stem)
logs["dep"] = logs.chemin.str.split("/").str[6]
morceaux = logs.nom.str.split("_")
logs["cle"] = morceaux.str[0] + "_" + morceaux.str[1]

DOM = {"971", "972", "973", "974", "975", "976", "977", "978"}
logs = logs[~logs.dep.isin(DOM)].reset_index(drop=True)
print(len(logs), "pages LOG de métropole")


def charger(chemin):
    with fs.open(chemin, "rb") as f:
        return Image.open(io.BytesIO(f.read())).convert("RGB")

129 pages LOG de métropole


# 5. La règle des cases

Identique au notebook 01. En mode A, c'est elle qui transforme les états vus par le modèle en réponse finale.


In [35]:
FORCE = ("cochee", "biffee", "coloree")
ETATS = {"vide", "cochee", "biffee", "coloree", "autre"}
MODALITE_BASSE = {"BI_Q10", "BI_Q11"}
BLANC_SI_PLUSIEURS = {"BI_Q9", "FL_Q3", "FL_Q7", "FL_Q13",
                      "FLDOM_Q3", "FLDOM_Q7", "FLDOM_Q11", "FLDOM_Q14",
                      "TPSRESALT", "AUTRECOMETU"}


def valeur_case(etat, gagnant):
    if etat == "vide":
        return "0"
    if gagnant is not None and etat == gagnant:
        return "1"
    if gagnant == "cochee" and etat == "biffee":
        return "2"
    return "3"


def regle_cases(question, etats):
    gagnant = next((f for f in FORCE if f in etats), None)
    detail = "".join(valeur_case(e, gagnant) for e in etats)
    positions = [i + 1 for i, c in enumerate(detail) if c == "1"]
    if not positions:
        return "", detail
    if question in BLANC_SI_PLUSIEURS and len(positions) > 1:
        return "", detail
    if question in MODALITE_BASSE:
        return str(min(positions)), detail
    return str(max(positions)), detail

# 6. Les questions posées au modèle

Les définitions des cinq états reprennent la section 1.2 du document de consignes. À relire avec les consignes : la formulation exacte compte beaucoup pour un modèle de langage.


In [36]:
PROMPT_OEIL = """Voici la page 4 d'une feuille de logement du recensement.
Regarde uniquement la question 1 « Type de logement ». Elle a six cases, dans cet ordre :
1 Maison, 2 Appartement, 3 Logement-foyer, 4 Chambre d'hôtel,
5 Habitation de fortune, 6 Pièce indépendante.

Pour chaque case, donne son état parmi :
- vide : aucune marque dans la case. Un grand trait qui traverse une partie du questionnaire compte comme vide.
- cochee : une croix dans la case, ou une case atteinte par une croix.
- biffee : case à demi cochée, par exemple un seul trait.
- coloree : case entièrement remplie.
- autre : case raturée, gribouillée, ou tout autre marquage.

Réponds uniquement avec une liste JSON de six états, dans l'ordre des cases.
Exemple : ["vide", "cochee", "vide", "vide", "vide", "vide"]"""


PROMPT_TOUT = """Voici la page 4 d'une feuille de logement du recensement.
Donne la réponse à la question 1 « Type de logement » : 1 Maison, 2 Appartement,
3 Logement-foyer, 4 Chambre d'hôtel, 5 Habitation de fortune, 6 Pièce indépendante.

Règles de saisie :
- Si une seule case est cochée, réponds son numéro.
- Si plusieurs cases sont marquées, le marquage le plus fort l'emporte :
  une croix, puis un demi-cochage, puis une case coloriée. Une case raturée ne compte pas.
- Si plusieurs cases ont ce marquage le plus fort, réponds le numéro le plus élevé.
- Si aucune case n'est marquée, réponds une chaîne vide.

Réponds uniquement en JSON : {"TYPL": "2"}"""

# 7. Test sur une seule image
On commence par les limites du notebook 01

In [37]:
CHEMIN_TEST = None     
import inspect
chemin = CHEMIN_TEST or logs.chemin.iloc[0]
page = charger(chemin)
print(chemin)

t0 = time.time()
brut_a = demander(page, PROMPT_OEIL)
etats = extraire_json(brut_a)
print(f"\nMODE A ({time.time() - t0:.1f} s)")
print("réponse brute :", brut_a)
if isinstance(etats, list) and len(etats) == 6 and set(etats) <= ETATS:
    print("CHOIX, DETAIL :", regle_cases("FL_Q1", etats))
else:
    print("réponse inutilisable")

t0 = time.time()
brut_b = demander(page, PROMPT_TOUT)
print(f"\nMODE B ({time.time() - t0:.1f} s)")
print("réponse brute :", brut_b)
print("TYPL :", (extraire_json(brut_b) or {}).get("TYPL"))

projet-mesure-qualite-rp/data/raw/RG25/BIE/99/01/009/0001/6901_5100000138_LOG.jpg

MODE A (1.2 s)
réponse brute : ["cochee", "vide", "vide", "vide", "vide", "vide"]
CHOIX, DETAIL : ('1', '100000')

MODE B (1.0 s)
réponse brute : {"TYPL": "1"}
TYPL : 1


# 8. Sur plusieurs documents

On commence par N = 20 


In [40]:


N = 200

lignes = []
t_debut = time.time()
for _, r in logs.head(N).iterrows():
    page = charger(r.chemin)
    ligne = {"cle": r.cle, "chemin": r.chemin}
    try:
        brut = demander(page, PROMPT_OEIL)
        etats = extraire_json(brut)
        if (isinstance(etats, list) and len(etats) == 6
                and all(isinstance(e, str) for e in etats) and set(etats) <= ETATS):
            ligne["etats_a"] = "|".join(etats)
            ligne["typl_a"], ligne["detail_a"] = regle_cases("FL_Q1", etats)
        else:
            ligne["typl_a"] = "INUTILISABLE"

        resultat_b = extraire_json(demander(page, PROMPT_TOUT))
        ligne["typl_b"] = (str(resultat_b.get("TYPL", "INUTILISABLE"))
                           if isinstance(resultat_b, dict) else "INUTILISABLE")
    except Exception as e:
        ligne["erreur"] = f"{type(e).__name__}: {e}"
    lignes.append(ligne)

vlm = pd.DataFrame(lignes)
duree = time.time() - t_debut
par_page = duree / max(len(vlm), 1) / 2      # 2 appels par page : mode A et mode B
print(f"{len(vlm)} documents en {duree:.0f} s")
print(f"modèle : {MODELE} (via l'API SSP Cloud)")
print(f"temps par page et par mode : {par_page:.1f} s")
print(f"300 pages LOG d'une campagne, un mode : {300 * par_page / 60:.0f} min")
if "erreur" in vlm:
    print(f"appels en erreur : {vlm['erreur'].notna().sum()}")
vlm.to_csv("resultats_vlm_typl.csv", index=False)
vlm.head()

129 documents en 316 s
modèle : qwen3-vl (via l'API SSP Cloud)
temps par page et par mode : 1.2 s
300 pages LOG d'une campagne, un mode : 6 min


,cle,chemin,etats_a,typl_a,detail_a,typl_b
0,6901_5100000138,projet-mesure-qualite-rp/data/raw/RG25/BIE/99/...,cochee|vide|vide|vide|vide|vide,1,100000,1
1,6901_5100000139,projet-mesure-qualite-rp/data/raw/RG25/BIE/99/...,vide|vide|vide|vide|vide|vide,,000000,
2,6901_5100000141,projet-mesure-qualite-rp/data/raw/RG25/BIE/99/...,vide|cochee|vide|vide|vide|vide,2,010000,2
3,6901_5100000145,projet-mesure-qualite-rp/data/raw/RG25/BIE/99/...,vide|cochee|vide|vide|vide|vide,2,010000,2
4,6901_5100000147,projet-mesure-qualite-rp/data/raw/RG25/BIE/99/...,vide|vide|vide|vide|vide|vide,,000000,


# 9. Comparaison au Fiqual

Même lecture et même clé que le notebook 01. TYPL est la colonne 17 du type 1.


In [39]:
def lire_brut(chemin, encodage="latin-1"):
    return pd.read_csv(chemin, sep="\x00", header=None, names=["ligne"], dtype=str,
                       encoding=encodage, keep_default_na=False, engine="python")["ligne"]


brut = lire_brut(f"s3://{BUCKET}/data/raw/{CAMPAGNE}/RP_PMQ_FIQUAL2599.txt")
champs = brut.str.split("|")
fq = pd.DataFrame(champs[champs.str[0] == "1"].tolist(), dtype=str)
fq["cle"] = fq[2].str.strip() + "_" + fq[4].str.strip()
print(fq.shape[0], "feuilles de logement dans le Fiqual")

comp = vlm.merge(fq[["cle", 17]].rename(columns={17: "fiqual"}), on="cle", how="inner")
comp["fiqual"] = comp.fiqual.str.strip()
for mode in ["a", "b"]:
    comp[f"ok_{mode}"] = comp[f"typl_{mode}"].fillna("").astype(str).str.strip() == comp.fiqual

print(f"\n{len(comp)} documents comparés")
print(f"mode A (l'œil + la règle) : {comp.ok_a.mean():.1%}")
print(f"mode B (le modèle fait tout) : {comp.ok_b.mean():.1%}")

120 feuilles de logement dans le Fiqual

20 documents comparés
mode A (l'œil + la règle) : 100.0%
mode B (le modèle fait tout) : 95.0%


# 9 bis. Enregistrer l'essai dans MLflow

Chaque essai est enregistre : modele, prompt, concordance, temps, memoire. On peut ainsi comparer les essais entre eux plus tard, sans les refaire.

Regle : si le prompt change, changer le nom du run (8B-prompt-v2).


In [29]:
import mlflow

mlflow.set_experiment("fiqual-typl")

# Fichiers à enregistrer
vlm.to_csv("resultats_vlm_typl.csv", index=False)
comp.to_csv("comparaison_fiqual.csv", index=False)
desaccords = comp.loc[~comp.ok_a | ~comp.ok_b]
desaccords.to_csv("desaccords.csv", index=False)

with mlflow.start_run(run_name=MODELE.split("/")[-1]):
    mlflow.log_params({"modele": MODELE, "N": len(comp),
                       "campagne": CAMPAGNE,
                       "execution": "api",
                       "api_url": BASE_URL})

    metriques = {"concordance_A": comp.ok_a.mean(),
                 "concordance_B": comp.ok_b.mean(),
                 "secondes_par_page": par_page,
                 "nb_desaccords": len(desaccords)}
    if "erreur" in vlm:
        metriques["taux_erreur_api"] = vlm["erreur"].notna().mean()
    mlflow.log_metrics(metriques)

    for fichier in ["resultats_vlm_typl.csv", "comparaison_fiqual.csv", "desaccords.csv"]:
        mlflow.log_artifact(fichier, artifact_path="resultats")

# Une seule fois, comme point de comparaison : la chaîne par densité du notebook 01
# with mlflow.start_run(run_name="densite"):
#     mlflow.log_params({"modele": "densite", "N": 120, "campagne": "RG25"})
#     mlflow.log_metrics({"concordance_A": 0.983})

# Les désaccords : c'est là que l'on apprend quelque chose
cols = ["cle", "fiqual", "typl_a", "etats_a", "typl_b"]
desaccords[[c for c in cols if c in desaccords.columns]]

🏃 View run qwen3-vl at: https://user-farisazibert-mlflow.user.lab.sspcloud.fr/#/experiments/3/runs/2ab5edbbcf3544d3946b56d953880ee9
🧪 View experiment at: https://user-farisazibert-mlflow.user.lab.sspcloud.fr/#/experiments/3


,cle,fiqual,typl_a,etats_a,typl_b
19,6901_5100000626,1,1,biffee|vide|vide|vide|vide|vide,2


In [30]:
import os
import mlflow

print("tracking URI :", mlflow.get_tracking_uri())
print("MLFLOW_S3_ENDPOINT_URL :", os.environ.get("MLFLOW_S3_ENDPOINT_URL"))
print("AWS_ACCESS_KEY_ID défini :", "AWS_ACCESS_KEY_ID" in os.environ)

for f in ["resultats_vlm_typl.csv", "comparaison_fiqual.csv", "desaccords.csv"]:
    print(f, "existe :", os.path.exists(f))

run = mlflow.last_active_run()
print("run id :", run.info.run_id)
print("artifact URI :", run.info.artifact_uri)

client = mlflow.MlflowClient()
print("artefacts :", [a.path for a in client.list_artifacts(run.info.run_id, "resultats")])

tracking URI : https://user-farisazibert-mlflow.user.lab.sspcloud.fr
MLFLOW_S3_ENDPOINT_URL : https://minio.lab.sspcloud.fr
AWS_ACCESS_KEY_ID défini : True
resultats_vlm_typl.csv existe : True
comparaison_fiqual.csv existe : True
desaccords.csv existe : True
run id : 2ab5edbbcf3544d3946b56d953880ee9
artifact URI : mlflow-artifacts:/3/2ab5edbbcf3544d3946b56d953880ee9/artifacts
artefacts : ['resultats/comparaison_fiqual.csv', 'resultats/desaccords.csv', 'resultats/resultats_vlm_typl.csv']


# 10. La page entière 

On donne toute la page et on demande toutes les questions d'un coup, en mode B.


In [28]:
COLONNES_HYPOTHESE = {
    17: "TYPL",  18: "ACHL",  19: "AACHL", 20: "ASCEN", 21: "NBPI",
    22: "SURF",  23: "STOC",  24: "HLML",  25: "AEMM",  26: "SANI",
    27: "CHFL",  28: "CMBL",  29: "VOIT",  30: "GARL",
}

PROMPT_PAGE = """Voici la page 4 d'une feuille de logement du recensement.
Pour chaque question, donne le numéro de la case cochée (1, 2, 3...), ou une chaîne vide
si rien n'est coché. Si plusieurs cases sont cochées, donne le numéro le plus élevé.
Pour les nombres écrits à la main, recopie les chiffres.

TYPL : question 1, type de logement
ACHL : question 2, année d'achèvement (numéro de la case)
AACHL : question 2, l'année écrite si « 2006 ou après » est cochée (4 chiffres)
ASCEN : question 3, ascenseur
NBPI : question 4, nombre de pièces (2 chiffres, par exemple 04)
SURF : question 5, surface (numéro de la case)
STOC : question 6
HLML : question 7, organisme HLM
AEMM : question 8, année d'emménagement (4 chiffres)
SANI : question 9, installations sanitaires
CHFL : question 10, moyen de chauffage
CMBL : question 11, combustible
VOIT : question 12, nombre de voitures
GARL : question 13, si elle existe sur la page

Réponds uniquement en JSON avec ces clés, toutes les valeurs entre guillemets."""


def normaliser(variable, valeur):
    v = str(valeur if valeur is not None else "").strip()
    if variable == "NBPI" and v.isdigit():
        v = v.zfill(2)
    return v


N_PAGE = 20
lignes = []
for _, r in logs.head(N_PAGE).iterrows():
    rep = extraire_json(demander(charger(r.chemin), PROMPT_PAGE, max_new_tokens=600)) or {}
    lignes.append({"cle": r.cle, **{k: normaliser(k, rep.get(k)) for k in COLONNES_HYPOTHESE.values()}})

page_vlm = pd.DataFrame(lignes)
ref = fq[["cle"] + list(COLONNES_HYPOTHESE)].rename(columns=COLONNES_HYPOTHESE)
c = page_vlm.merge(ref, on="cle", suffixes=("_vlm", "_fiqual"))

print(f"{len(c)} pages comparées\n")
for var in COLONNES_HYPOTHESE.values():
    ok = c[f"{var}_vlm"] == c[f"{var}_fiqual"].map(lambda x: normaliser(var, x))
    print(f"{var:6} {ok.mean():6.1%}")

20 pages comparées

TYPL    90.0%
ACHL    65.0%
AACHL   65.0%
ASCEN   75.0%
NBPI    65.0%
SURF    75.0%
STOC    80.0%
HLML    70.0%
AEMM    80.0%
SANI    75.0%
CHFL    60.0%
CMBL    50.0%
VOIT    65.0%
GARL    75.0%
